---
title: Week 4.6, Group Project Analysis of a leachate dataset
subject: Group Project ECTB1230 2026
subtitle: Example notebook for Day 1
authors:
  - name: Timo Heimovaara
    affiliations:
      - Delft University of Technology, department of Geoscience & Engineering
    orcid: 0000-0003-4230-7476
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-05-20
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Introduction

Within the Netherlands we have been carrying out a project called "Introduction of Sustainable Aftercare of Landfills" or in Dutch: Introductie Duurzaam Stortbeheer [iDS](https://duurzaamstortbeheer.nl/). The aim of this project is to investigate if we can reduce the emission potential of a wastebody by active treatment using infiltration of water aeration. Infiltration is hypothesized to increase the flushing of dissolved contaminants from the wastebody, aeration is hypothesized to simulate the biodegradation of organic matter in the waste body. Both approaches will eventually lead to lower contaminant concentrations in the leachate.

Sanitary engineered landfills are technical facilities which allow us to store waste indefinitely. In order to protect the environment and human health from emissions, these sites are fitted with impermeable bottom liners with a drainage systems for leachate collection, and gas extraction systems within the waste body for collection of methane and other greenhouse gases being produced in the waste body. After the landfilling has been completed the current regulations require the wastebody to be capped with a water tight liner so that no rainfall can infiltrate. As a result, the driving force for leachate emissions is no longer present. The drawback, however, is that this coverliner needs to be replaced every 75 years. This is nicely illustrated with the animation you can find on [iDS](https://duurzaamstortbeheer.nl/) at the bottom of the page.

Pilot projects are being carried out within the context of iDS at three landfills: Kragge near Bergen op Zoom, Braambergen near Almere and Wieringermeer near Wieringermeer. A large amount of background information can be found in the ["background"](https://duurzaamstortbeheer.nl/achtergrond/) section of the iDS website. Here you can find information on the three pilot projects in the section ["Project documents"](https://duurzaamstortbeheer.nl/projectstukken/) and a number of [publications](https://duurzaamstortbeheer.nl/publicaties/) that have been written in the course of the project. 

Leachate quality from the three pilot projects have been measured with a relatively high frequency since 2012, the start of the base-line monitoring. Preparation of the active treatment began in 2016 and the treatment was started in 2018 and will continue until 2029. 

Your task is to analyse a leachate data set in order to answer the following questions:
1. What is the likely composition of the leachate within the waste body?
2. How much solids will precipitate from the leachate once it is exposed to air in the water treatment plant?
3. How much solids will have preciptated as the leachate moved from the bulk of the waste to the water treatment system?
4. Do these processes vary over time.

In the ["Project documents"](https://duurzaamstortbeheer.nl/projectstukken/) section of the iDS website, you can find the original plans for the pilot projects. The general overview of the iDS projects is given in the ["Integraal Plan van Aanpak"](https://duurzaamstortbeheer.nl/wp-content/uploads/2023/09/IENM-BSK-2014-116919-Def-concept-IPvA-versie-mei-2014.pdf). The site specific plans can be found in the documents starting with "Deel van Aanpak". You can also find two documents in English giving similar information: "Project plan Sustainable Landfill Management...".

## Project assignment
The assignment that you need to do is to carry out an analysis of a dataset that is provided to you. In many cases where routine interpretations are done, it is most efficient to start with a pre-existing approach or script for your analysis and then modify the approach to your specific needs. This is also the approach you need to do here. The notebooks provided to you, show a similar analysis so you can apply this notebook as your template. 

Your responsibility is to adapt the notebook and the interpretation where necessary. At the same time you need to understand what you are doing. The generated results need to be included in your final report.


## Project actvities Day 1

Before you can start working on analysing the dataset, you have to familiarize yourself with the data provided to you. The data is provided in an Excel sheet. The data is an export from a single waste body from one of the pilot projects and contains the date a sample was taken and the results of a chemical analysis in the laboratory.

First you need to obtain an overview of the data. Your assignment for this day is to:
1. Import the data;
2. Get a quick over view of the content and the structure of the dataset;
3. Understand how to plot the time series in the data set, save the figures to a file and create an overview report;
4. Need to calculate molar concentrations from mg/l values;
5. Think about what questions you want to resolve with this data?
    - Saturation status of the samples as they are;
    - What were mostly likely conditions where the samples originated?
    - What will happen to the samples if the leachate would be discharged to a system at atmospheric conditions.
5. Prepare the interface to PyOrchestra, have a look at provided GUI of Orchestra and the corresponding Chemistry File

All steps you need to do have been shown earlier in weeks 3.7 and 4.2. To help you get started I have provided this notebook where I show how to carry out the first 4 steps. You need to create your own notebook in order to analyse the dataset provided to you.
To do this you create a new Notebook in Jupyter-lab. You can copy the python cells below to your own notebook. Please make sure that you adapt the cells where necessary so that it matches your own data.

The data is provided as an Excel Workbook. I assume you will use pandas in order to process the data and seaborn to plot the output. In the following text I will provide you with some hints how to process the data using pandas.

In order to import these using pandas, you need to need install openpyxl:  
- mamba install -c conda-forge openpyxl

I suggest that have the [Python Data Sicence Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) by Jake Vanderplas available in order to have a better grip of the python concepts. This book gives an introduction on NumPy, Pandas, Matplotlib (with a section on seaborn). These are probably the main packages you will be using during your time at the TU Delft. 

First we import the required python libraries.

In [1]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

# The figures will be shown in a separate window, not inline in the notebook 
%matplotlib qt 
sns.set() 


In [2]:
# This code section is required for the Jupyter Book to find the input files for the 
# Orchestra simulations. The input files are located in the same directory as this notebook, 
# but when running the notebook in Jupyter Book, the current working directory is the book root, 
# not the directory of this notebook. Therefore, we need to find the path to the book root and 
# then construct the path to the input files from there.

def find_book_root(start: Path | None = None) -> Path:
    """
    Walk upward from `start` (or CWD) until a directory containing Jupyter Book
    marker files is found. Returns the path to the book root.
    Raises FileNotFoundError if no root is found.
    """
    config_any = {"_config.yml", "_config.yaml"}      # some projectrs use .yaml
    myst_any = {"myst.yml", "myst.yaml"}            # jupyter-book uses _toc.yml

    cur = Path(Path.cwd()).resolve()

    for parent in [cur, *cur.parents]:
        children = {f.name for f in parent.iterdir()} if parent.exists() else set()
        has_any_config = bool(config_any & children)
        has_any_myst = bool(myst_any & children)
        if has_any_config and has_any_myst:
            return parent

    raise FileNotFoundError(
        f"Could not find Jupyter Book root (no _config.y* and myst.y* found above {cur})"
    )


def path_from_book_root(*parts: str | Path) -> Path:
    root = find_book_root()
    p = (root.joinpath(*parts)).resolve()
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p

# In order for orchestra to run we need to change directory to the directory with the input file:

orchestra_path = path_from_book_root("content", "project_THe", "Orchestra_Project")
# print(orchestra_path)

# Please note: When you are running the notebook from the folder 
# with the Orchestra files use the next line:
# orchestra_path = '.'

### Dataset  

This data set is an export from the iDS database of the iDS project carried out by the Dutch Sustainable Landfill foundation.

The exported data contains the the results from laboratory analyses of 
the following parameters
[
    'pH', 'Natrium [Na]', 'Kalium [K]', 'Calcium [Ca]', 
    'Magnesium [Mg]', 'IJzer [Fe]', 'Mangaan [Mn]', 
    'Ammonium (als NH4)', 'Fosfaat (als PO4)',
    'Bicarbonaat', 'Chloride', 'Sulfide', 
    'Sulfaat (als SO4)', 
    'Temperatuur',
]

This list contain the macro parameters (where in general the concentration in the leachate is above 1 mg/l).

The data has been preprocessed in order to remove obvious outliers.

As a consequence of the measurement strategy and the outlier removement, the number of analyses on the samples taken over time varies significantly.

Measurement strategy:
- originally: 1 to 2 samples per year;
- iDS: 26 samples per year, parameters vary per frequency
    - parameters 26 times per year (bi weekly)
    - parameters 13 times per year (every four weeks)
    - parameters 4 times per year  (quarterly)
    - parameters 1 time per year (yearly)
    - parameters that are taken irregurlary

The details of the measurement frequencies can be found in the project plans on the [iDS projects](https://duurzaamstortbeheer.nl/projectstukken/) website.


We first import the data and print the some rows in order to see the data stored in the spreadsheet. You can also have a look at the spreadsheet it self using Excel or something similar. Please note, do not edit the raw data file.


In [3]:
# %% 1
# import the data set from the excel file.
# We will use the data from the leachate monitoring at PP-11N.

df_leachate = pd.read_excel('data/df_macros_PP-11Z.xlsx')

# check the column names
print(f'The dataset has the following columns: {list(df_leachate.columns)}')
# check the number of rows and columns
print(f'The dataset has {df_leachate.shape[0]} rows and {df_leachate.shape[1]} columns.')

# check the first few rows of the dataset
print(df_leachate.head(10))

# check the last few rows of the dataset
print(df_leachate.tail(10))


The dataset has the following columns: ['compartment', 'measpointname', 'date', 'cname', 'val_mgl', 'uname_mgl']
The dataset has 2265 rows and 6 columns.
  compartment measpointname       date              cname  val_mgl uname_mgl
0       BB11Z        PP-11Z 2000-11-07  Sulfaat (als SO4)    780.0      mg/l
1       BB11Z        PP-11Z 2000-11-07                 pH      6.8         -
2       BB11Z        PP-11Z 2003-07-15                 pH      7.1         -
3       BB11Z        PP-11Z 2003-07-15           Chloride    690.0      mg/l
4       BB11Z        PP-11Z 2003-07-15  Sulfaat (als SO4)    820.0      mg/l
5       BB11Z        PP-11Z 2004-06-08        Temperatuur     20.0        °C
6       BB11Z        PP-11Z 2004-06-08           Chloride    560.0      mg/l
7       BB11Z        PP-11Z 2004-06-08                 pH      6.9         -
8       BB11Z        PP-11Z 2004-06-08  Sulfaat (als SO4)    540.0      mg/l
9       BB11Z        PP-11Z 2005-06-24  Sulfaat (als SO4)   1200.0      mg/l

This data set is a so-called list type. We can access the data using the *cname* and/or *date* column for further processing. The earliest available data are from 2000, the latest are for the end of April 2026. For most dates, multiple chemical components hvae been analysed which means that we can use the date to select a "sample" on which multiple parameters have been measured.



### Step 1: Plotting the timeseries for each chemical component in the dataset
The first thing to do is to get an overview of the data present. As you can see have 2240 rows with data points. Each row represents a laboratory analysis result from a sample with a specific date, for a specific component. We do not know which components are present and how many analyses are present for each sample.
A quick way to obtain the required overview is to generate a series of figures and look at these. 
- We create a separate folder to store these figures: ./Figures;
- We plot time series for each component in separate figures labelled by *cname*;

In [4]:
# Use seaborn line plot to plot the concentration of each component over time. 
# Use the date as x-axis and the concentration as y-axis. Use different colors 
# for different components. Add a legend to the plot.

plt.close('all')   # close all previous figures (good practice when running a new code)

# Generate a list of all unique chemical components in the dataset
component_list = df_leachate['cname'].unique()   # get the unique component names

# Prepare for plotting, it is good practice to have the figure and axis handles
# available for annotating the plots
figs = []   # initialize a counter for the figure index
axs = []   # initialize a counter for the axis index

# We will run a loop over all components in component_list, but we also
# need a counter to access the figs or axs list. We initalize this to -1 so
# after the first-update it starts with zero
ii = -1   # initialize a counter for the component index

# Loop over all components in component_list
for cn in component_list:
    ii += 1
    fig, ax = plt.subplots()  # create a new figure and axis for each component
    figs.append(fig)
    axs.append(ax)
    sns.lineplot(
        data=df_leachate[df_leachate['cname'] == cn],
        x='date',
        y='val_mgl',
        ax=axs[ii],
        marker='o',        
        )
    axs[ii].set_title(f'Concentration of {cn} over time')
    axs[ii].set_xlabel('Date') 
    axs[ii].set_ylabel('Concentration (mg/L)')

    # write figure to a file in the local Figures folder. The name is defined by cn
    fig.savefig(f'Figures/{cn}_concentration_over_time.png', dpi=300, bbox_inches='tight')


In [5]:
# We have saved the figures to the Figures folder so we can close them.
# The figures are in *.png format which you can easily import in to your report 
plt.close('all')

In [38]:
# Use seaborn  violin plot to plot the concentration of each component over time. 
# Use the date as x-axis and the concentration as y-axis. Use different colors 
# for different components. Add a legend to the plot.

# Add a classification to the data set to identify, spring (march april may) summer(june, july August), 
# autumn (Sept Oct Nov) and winter (Dec, Jan, Feb).


plt.close('all')   # close all previous figures (good practice when running a new code)

# Generate a list of all unique chemical components in the dataset
component_list = df_leachate['cname'].unique()   # get the unique component names

# Prepare for plotting, it is good practice to have the figure and axis handles
# available for annotating the plots
figs = []   # initialize a counter for the figure index
axs = []   # initialize a counter for the axis index

# We will run a loop over all components in component_list, but we also
# need a counter to access the figs or axs list. We initalize this to -1 so
# after the first-update it starts with zero
ii = -1   # initialize a counter for the component index

# Loop over all components in component_list
for cn in ['Chloride']: #component_list:
    ii += 1
    fig, ax = plt.subplots()  # create a new figure and axis for each component
    figs.append(fig)
    axs.append(ax)
    sns.violinplot(
        data=df_leachate[df_leachate['cname'] == cn],
        # x='date',
        y='val_mgl',
        ax=axs[ii],
        # bins = 20,        
        )
    axs[ii].set_title(f'Concentration of {cn} over time')
    #axs[ii].set_xlabel('Date') 
    #axs[ii].set_ylabel('Concentration (mg/L)')

    # write figure to a file in the local Figures folder. The name is defined by cn
    fig.savefig(f'Figures/{cn}_concentration_over_time.png', dpi=300, bbox_inches='tight')


In [34]:
component_list

<ArrowStringArray>
[ 'Sulfaat (als SO4)',                 'pH',           'Chloride',
        'Temperatuur',       'Calcium [Ca]',       'Mangaan [Mn]',
         'Kalium [K]',      'Silicium [Si]', 'Ammonium (als NH4)',
       'Natrium [Na]',        'Bicarbonaat',     'Magnesium [Mg]',
            'Sulfide',         'IJzer [Fe]',  'Fosfaat (als PO4)']
Length: 15, dtype: str

```{note}
**Give an intepretation of the concentration time series**
The reason for collecting samples over time is that the landfill shows a dynamic behavior which is driven by seasonal changes. One of the most notciable is the seasonal change in the leachate production which is related to the net precipitation. Net precipitation is the difference between the rainfall and the evapo-transpiration from the cover-layer of the waste body. The evapo-transpiration in summer is much larger than in the winter, it is so high on many days the evpo-transpiration exceeds rainfall by far (there are many days where it does not rain and plants still evaporate large amounts of water). As a result leachate production in summer is much smaller than in the winter.

Understanding this, can you explain the patterns you see in the leachate concentration variations over time. For this interpretation, consider the possible reactions a chemical component can undergo.
```

### Step 2: Use pyOrchestra to analyse a sample with the most parameters in the dataset

In order to carry out a geochemical analysis we need to have the following information:
- Which chemical components will we used as our master-species?
- Which samples in the data-set have largest number of chemical components analysed?

For the first question the answer is already available. In the above code we have created a list of unique components. We can print this list.

In [6]:
print(component_list)

<ArrowStringArray>
[ 'Sulfaat (als SO4)',                 'pH',           'Chloride',
        'Temperatuur',       'Calcium [Ca]',       'Mangaan [Mn]',
         'Kalium [K]',      'Silicium [Si]', 'Ammonium (als NH4)',
       'Natrium [Na]',        'Bicarbonaat',     'Magnesium [Mg]',
            'Sulfide',         'IJzer [Fe]',  'Fosfaat (als PO4)']
Length: 15, dtype: str


The reason for us to require a sample that has measurements of as many parameters as possible, is that we then can assess the complete chemical system for which we have data. To find the dates in the data-set with most analyses, we use some tools available in pandas. The background for these tools are clearly described in the [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/03.08-aggregation-and-grouping.html).


We use the concept of data aggregation or grouping. Our aim is to creating a new dataframe from the data containing the number of parameters per sampling date.

We use methods built in the pandas library which are associated with the pandas dataframe:
- count the number of parameters (identified in the columne *cname*) by grouping over *measurementpointname* and *date*;
- using count() as the aggregation function;
- renaming the output column to *macro_count*
- and then finally sorting the data frame using the values in *macro_count* in descending order

The first record in this data frame contains the sample, identifed by *date* with the most parameters. The top rows in this table contain the samples with most parameters, the bottom rows the samples with the least number of parameters.

We will proceed with the date with the most parameters to do the first analysis.

In [7]:
# %% 2
# Check which dates have most parameters measured.
# We will use these parameters as our input for the Orchestra 
# calculation.

# We can use the groupby function in pandasto group the data by date 
# and count the number of parameters measured for each date.
# count() creates a new column with the count of the number of parameters
# measured for each date which we will call macro_count using reset_index 
# to create a new dataframe with the macro_count column and sort the values 
# by macro_count in descending order.

df_par_counts = (
    df_leachate.groupby(['measpointname', 'date'])['cname']
    .count()
    .reset_index(name='macro_count')
    .sort_values('macro_count', ascending=False)
)

# The first value is a sample (date) with the most parameters measured
date_with_most_pars = df_par_counts.iloc[0]['date']

print(df_par_counts.head(10))

print(f"The date with the most parameters measured is: {date_with_most_pars}")


    measpointname       date  macro_count
280        PP-11Z 2023-12-12           15
66         PP-11Z 2014-01-21           15
290        PP-11Z 2024-04-25           14
70         PP-11Z 2014-03-18           14
75         PP-11Z 2014-10-07           14
227        PP-11Z 2021-12-20           14
170        PP-11Z 2019-08-27           14
56         PP-11Z 2013-09-17           13
48         PP-11Z 2013-05-28           13
265        PP-11Z 2023-05-02           13
The date with the most parameters measured is: 2023-12-12 00:00:00


We see that the maximum number of components in the data set is 15 for two dates, then 14 for five dates and then 13. For our first analysis we will work with one of the two data sets with 15 analysis results. 

### Calculate the concentrations in mol/l
The analysis results are given in mg/l, for the geochemical calculations with Orchestra we require the units to be in mol/l, we therefore need a conversion table to do this. At the same time, we require a translation of the component names to those used in the pyOrchestra library. We do both by creating a python dictionary.

Please note, that if your data-set has additional parameters, you need to adjust the following dataframe accordingly.


In [8]:
# componentname, molar mass (g/mol), Orchestra parameter name
conversion_table = pd.DataFrame(
    data={
        'Sulfaat (als SO4)': [96.06, 'SO4-2.tot'],
        'Sulfide': [32.07, 'S-2.tot'],
        'Natrium [Na]': [22.99, 'Na+.tot'],
        'IJzer [Fe]': [55.85, 'Fe+2.tot'],
        'Magnesium [Mg]': [24.31, 'Mg+2.tot'],
        'Calcium [Ca]': [40.08, 'Ca+2.tot'],
        'Ammonium (als NH4)': [18.04, 'NH4+.tot'],
        'Chloride': [35.45, 'Cl-.tot'],
        'Bicarbonaat': [61.02, 'HCO3-.tot'],
        'Fosfaat (als PO4)': [94.97, 'PO4-3.tot'],
        'Kalium [K]': [39.10, 'K+.tot'],
        'Silicium [Si]': [28.09, 'Si.tot'],
        'Mangaan [Mn]': [54.94, 'Mn+2.tot'],
        'Temperatuur': [1e-3, 'T'], 
        'pH': [1e-3, 'pH'], # please note the factor 1e-3 which will be corrected for in the conversion to moles/l.
    }, 
    index=['molar_mass','orchestra_parameter']).T
    


In [9]:
print(conversion_table)


                   molar_mass orchestra_parameter
Sulfaat (als SO4)       96.06           SO4-2.tot
Sulfide                 32.07             S-2.tot
Natrium [Na]            22.99             Na+.tot
IJzer [Fe]              55.85            Fe+2.tot
Magnesium [Mg]          24.31            Mg+2.tot
Calcium [Ca]            40.08            Ca+2.tot
Ammonium (als NH4)      18.04            NH4+.tot
Chloride                35.45             Cl-.tot
Bicarbonaat             61.02           HCO3-.tot
Fosfaat (als PO4)       94.97           PO4-3.tot
Kalium [K]               39.1              K+.tot
Silicium [Si]           28.09              Si.tot
Mangaan [Mn]            54.94            Mn+2.tot
Temperatuur             0.001                   T
pH                      0.001                  pH


The data provided contains information for 15 parameters of which are all present in the dataset for the two dates with most parameters.

Other samples in the data set have less parameters. For the further analysis on samples with less analysed components, we would like to include these parameters in the analysis as well. 

In order to handle the missing values we aim to use the mean value from the total dataset. We obtain the mean values from the data set with the data aggregation methods available in pandas using a similar approach as above for obtaining df_par_counts. In addition to calculating the average concentration in mg/l we also calculate the concentration in mol/l using the information present in the conversion_table dataframe.

In [10]:
# We will calculate the average concentration in mg/l for each component and add it to the conversion table.

df_comp_means = (
    df_leachate.groupby('cname')['val_mgl']
    .mean()
    .reset_index(name='avg_val_mgl')
    .set_index('cname')
)

#print(df_comp_means)

# We add the avg_val_mgl to the conversion table 
# for the components that are in the conversion table.
conversion_table = (
    conversion_table.merge(
        df_comp_means, 
        left_index=True,
        right_index=True,
        how='left')
)

# calculate the average concentration in moles/l for each component
conversion_table['avg_val_mol_l'] = conversion_table['avg_val_mgl'] / conversion_table['molar_mass'] * 1e-3

print(conversion_table)

                   molar_mass orchestra_parameter  avg_val_mgl avg_val_mol_l
Sulfaat (als SO4)       96.06           SO4-2.tot  1054.721402       0.01098
Sulfide                 32.07             S-2.tot     1.582291      0.000049
Natrium [Na]            22.99             Na+.tot   578.793103      0.025176
IJzer [Fe]              55.85            Fe+2.tot    12.462162      0.000223
Magnesium [Mg]          24.31            Mg+2.tot   217.396364      0.008943
Calcium [Ca]            40.08            Ca+2.tot   619.587500      0.015459
Ammonium (als NH4)      18.04            NH4+.tot   127.229601      0.007053
Chloride                35.45             Cl-.tot   862.349515      0.024326
Bicarbonaat             61.02           HCO3-.tot  2605.889328      0.042705
Fosfaat (als PO4)       94.97           PO4-3.tot     9.502385        0.0001
Kalium [K]               39.1              K+.tot   253.137255      0.006474
Silicium [Si]           28.09              Si.tot   266.635588      0.009492

### Select data from data-set for further analysis
In the above code sections we have:
- imported the raw data;
- created plots to see the time-series for all available components;
- created a translation table with information necessary for calculating the molar concentrations and the component names used in pyOrchestra;
- identified which sampling-dates have the most chemical analyses available.

Now we are ready to extract the information from the selected date with most chemical analyses. Then we need to add a columns with the molar concentrations and orchestra component (or parameter) names.

In [11]:
# %%
# We use the date with most parameters for our first analysis
sel_idx = df_leachate['date'] == date_with_most_pars

df_work = df_leachate[sel_idx].copy()



# Export df_work to an Excel file so that we can 
# have a quick access to the parameters in it
# for setting up the translation from mg/l to moles/l 
# for the Orchestra input.
# %%
#df_work.to_excel('tmp/df_work_PP-11N.xlsx', index=False)

# %%
# Using the content from the file we now create a table
# with the parameters, their values and the conversion to moles/l.
# We will use this table to set up the translation from mg/l to moles/l
# for the Orchestra input.


# We can now use this conversion table 
# to convert the values in the df_work dataframe
# from mg/l to moles/l and to add a new column with Orchestra parameter names.

df_work['val_mol_l'] = df_work.apply(
    lambda row: 
        (row['val_mgl'] * 1e-3) / conversion_table.loc[row['cname'],'molar_mass'] 
        if row['cname'] in conversion_table.index else row['val_mgl'], axis=1)

df_work['orchestra_param'] = df_work.apply(
    lambda row: 
        conversion_table.loc[row['cname'], 'orchestra_parameter']
        if row['cname'] in conversion_table.index else row['cname'],axis=1
    )
    
#select temperatures and add 273.15 to convert to K
sel_temp = df_work['orchestra_param'] == 'T'
df_work.loc[sel_temp, 'val_mol_l'] += 273.15


In [12]:
print(df_work)
        

     compartment measpointname       date               cname      val_mgl  \
1874       BB11Z        PP-11Z 2023-12-12                  pH     6.810000   
1875       BB11Z        PP-11Z 2023-12-12  Ammonium (als NH4)    60.686157   
1876       BB11Z        PP-11Z 2023-12-12   Sulfaat (als SO4)  1415.281347   
1877       BB11Z        PP-11Z 2023-12-12        Natrium [Na]   290.000000   
1878       BB11Z        PP-11Z 2023-12-12          Kalium [K]   130.000000   
1879       BB11Z        PP-11Z 2023-12-12        Calcium [Ca]   650.000000   
1880       BB11Z        PP-11Z 2023-12-12      Magnesium [Mg]   170.000000   
1881       BB11Z        PP-11Z 2023-12-12        Mangaan [Mn]     1.200000   
1882       BB11Z        PP-11Z 2023-12-12            Chloride   250.000000   
1883       BB11Z        PP-11Z 2023-12-12         Bicarbonaat  1600.000000   
1884       BB11Z        PP-11Z 2023-12-12       Silicium [Si]    14.300000   
1885       BB11Z        PP-11Z 2023-12-12          IJzer [Fe]   

### Add missing values to the data set
Although not required for this first analysis, when analysing samples with less analyses than 15, you need use the averaged value calculated above for the missing components. The following code section shows how to do this.

Once we have completed the processing of the sample we aim to analyse with pyOrchestra we save it to a spreadsheet in the local *tmp* directory. This allows us to easily set-up the Orchestra input files using the Orchestra GUI in the *orchestra2026.jar* file. For the current example this preparation already has been done, and if you do not have to add additional chemical components to the problem, you can run your own notebook using these Orchestra files as well.

To check the results we print the final processed data set.

In [13]:

# We need to add the missing parameters to df_work so that we have all the parameters in the conversion table in df_work.

# First we make cname the index of df_work so we can easily add the missing parameters
df_work.set_index('cname', inplace=True)

for cn in conversion_table.index:
    if cn not in df_work.index:
        df_work.loc[cn,'val_mol_l'] = conversion_table.loc[cn, 'avg_val_mol_l']
        df_work.loc[cn,'orchestra_param'] = conversion_table.loc[cn, 'orchestra_parameter']

print(df_work)

# %%
# Rewrite df_work to excel so we can copy the contents to the Orchestra input file.
df_work.to_excel('tmp/df_work_PP-11N.xlsx', index=False)
# %%


                   compartment measpointname       date      val_mgl  \
cname                                                                  
pH                       BB11Z        PP-11Z 2023-12-12     6.810000   
Ammonium (als NH4)       BB11Z        PP-11Z 2023-12-12    60.686157   
Sulfaat (als SO4)        BB11Z        PP-11Z 2023-12-12  1415.281347   
Natrium [Na]             BB11Z        PP-11Z 2023-12-12   290.000000   
Kalium [K]               BB11Z        PP-11Z 2023-12-12   130.000000   
Calcium [Ca]             BB11Z        PP-11Z 2023-12-12   650.000000   
Magnesium [Mg]           BB11Z        PP-11Z 2023-12-12   170.000000   
Mangaan [Mn]             BB11Z        PP-11Z 2023-12-12     1.200000   
Chloride                 BB11Z        PP-11Z 2023-12-12   250.000000   
Bicarbonaat              BB11Z        PP-11Z 2023-12-12  1600.000000   
Silicium [Si]            BB11Z        PP-11Z 2023-12-12    14.300000   
IJzer [Fe]               BB11Z        PP-11Z 2023-12-12     8.10

### Develop a strategy how you will analyse this sample with pyOrchestra...

The next steps which will mainly be carried out on Day 2 of this assignment, is to use the processed data you collected above in a series of pyOrchestra calculations. This requires you to revist the material from the previous weeks in this course:

1. What types of reactions do you expect to happen between the chemical components in the sample?
2. Which of these reactions take place in the water phase only?
3. Which of these reactions take place between the water phase and gas phase?
4. Which of these reactions take place between the water phase and solid phase?

In answering these questions, consider possible gases and minerals that may develop. The content from weeks 3.7 and 4.2 cover most of the material you need to figure this out.

Using the information you gathered you need to:
1. Choose which chemical compounds you use a your master species;
2. Check the chemistry.inp file with the Orchestra GUI so that it matches your choice of master species;
3. We need to answer 3 questions: 
    - what is the initial state of the sample?, 
    - what happens if it is equilibrated with the atmosphere?, 
    - and what were the conditions where it originated? 
4. What type of results do you expect to get from your pyOrchestra calculation, how will you interprete these results?
5. Start working on your report with all of the above information and think how your report should include your interpretation?

## Day 2: Assessing the water samples

### Project actvities Day 2

On the first day you looked at the data provided to you and selected a data from a single sample on which you will do your first analysis. After that you have been thinking about a strategy to analyse the data and make some interpretations. On the second day you will carry out a full interpretation of the selected sample.

The steps you need to do on this day are:
1. Prepare the interface to PyOrchestra, have a look at provided GUI of Orchestra and the corresponding Chemistry File;
2. Decide on how many Chemistry files you require for your analysis;
3. Implement the input and output arrays required to interface with pyOrchestra for each Chemistry file;
4. Decide on which output you want to use from pyOrchestra for your interpretation
5. Put all this together in code for pyOrchestra calculations;
6. Run three types of pyOrchetra calcautions to answer the three questions:
    - what is the initial state of the sample?, 
    - what happens if it is equilibrated with the atmosphere?, 
    - and what were the conditions where it originated? 
7. Interpret the results;
8. Include the outcomes of your work in your report;

All steps you need to do have been shown earlier in weeks 3.7 and 4.2. 

### Setting up pyOrchestra
In order solve this problem with pyOrchestra we first initialize our problem using the *chemistry_Travertine.inp* file. 
This file predefines the aqueous chemical system in such a way that we can use the information from the tables in the paper as inputs to the Orchestra simulation.

Once pyOrchestra is initialized, running a simulation consists of a series of steps where the values of the required set of input variables are passed via *InVARS* to ORCHESTRA, after which a set of corresponding output variables are passed back in *OutVars*.

ORCHESTRA is initialized in pyOrchestra using the inputfile *chemistry_Travertine.inp*, created above with the ORCHESTRA-GUI. 

After initialization in Python, we know which variables will be passed through *OutVars* and can be used in *InVars*.

Using the *Output selector* tab on the Chemistry section of the GUI, we can generate lists that we can use to evaluate the output from the simulation.The following code stores output for the logactivity (_*.logact_), and molar concentrations (_*.con_) for all species in the model, we also export the saturation indices for all possible minerals in the system (_*.SI_) and the corresponding total amounts of minerals in the system (_*.tot_) and we export the totals of the gas phases. Using separate output lists for the minerals makes interpretation more efficient later.

Finally we also define the master species in the _InVars_ list, where we also include the constants in the model we want to be able to change during the simulation.

```{note}
For our scenarios it will be necessary to be able to fix the CO2[g] pressure to a constant value. In order to be able to do this we need to set the variables *gas_type* and *fixed_logact_CO2* during the simulation. We therefore include these in the _InVars_ list.
```


In [14]:
# We get the list of primary states from df_work and the corresponding parameter names in
# the Orchestra input file.

# We obtain the required outputs for the dissolved species using the OrchestraGUI

out_logact = [
    'Alkalinity.logact','Anhydrite[s].logact','Ar.logact',
    'Aragonite[s].logact','C.logact','CO2.logact','CO2[g].logact','CO2g[s].logact',
    'CO3-2.logact','C[+4].logact','Ca.logact','Ca+2.logact','CaCO3.logact',
    'CaHCO3+.logact','CaOH+.logact','CaSO4.logact','Ca[HPO4].logact','Ca[OH]+.logact',
    'Ca[SO4].logact','Calcite[s].logact','Cl.logact','Cl-.logact','Dolomite[s].logact',
    'Fe.logact','Fe+2.logact','FeCO3.logact','FeCl+.logact','FeCl2.logact','FeCl3-.logact',
    'FeS[ppt][s].logact','Fe[+2].logact','Fe[CO3]2-2.logact','Fe[H2PO4]+.logact',
    'Fe[HPO4].logact','Fe[HS]+.logact','Fe[HS]2.logact','Fe[NH3]+2.logact',
    'Fe[NH3]2+2.logact','Fe[NH3]4+2.logact','Fe[OH]+.logact','Fe[OH]2.logact',
    'Fe[OH]3-.logact','Fe[OH]4-2.logact','Fe[SO4].logact','Gypsum[s].logact',
    'H.logact','H+.logact','H2CO3.logact','H2O.logact','H2O[g].logact','H2S.logact',
    'H2[PO4]-.logact','H2[SiO4]-2.logact','H3[PO4].logact','H3[SiO4]-.logact',
    'H4[SiO4].logact','HCO3-.logact','HPO4-2.logact','HS-.logact','HSO4-.logact',
    'H[+1].logact','Halite[s].logact','Hydroxyapatite[s].logact','K.logact',
    'K+.logact','KPO4-2.logact','KSO4-.logact','K[HPO4]-.logact',
    'Mackinawite[s].logact','Melanterite[s].logact','Mg.logact','Mg+2.logact',
    'MgCO3.logact','MgHCO3+.logact','MgOH+.logact','MgSO4.logact','Mg[H2PO4]+.logact',
    'Mg[H3SiO4]+.logact','Mg[HPO4].logact','Mg[NH3]+2.logact','Mg[NH3]2+2.logact',
    'Mg[NH3]3+2.logact','Mg[NH3]4+2.logact','Mg[PO4]-.logact','Mn.logact',
    'Mn+2.logact','Mn2[OH]+3.logact','Mn2[OH]3+.logact','MnCl+.logact','MnCl2.logact',
    'MnCl3-.logact','Mn[CO3].logact','Mn[HCO3]+.logact','Mn[HPO4].logact',
    'Mn[HPO4]2-2.logact','Mn[NH3]+2.logact','Mn[NH3]2+2.logact','Mn[OH]+.logact',
    'Mn[OH]2.logact','Mn[OH]3-.logact','Mn[OH]4-2.logact','Mn[SO4].logact','NH3.logact',
    'NH4+.logact','Na.logact','Na+.logact','NaCO3-.logact','NaH2PO4.logact',
    'NaHCO3.logact','NaPO4-2.logact','NaSO4-.logact','Na[HPO4]-.logact','O.logact',
    'OH-.logact','O[-2].logact','PO4-3.logact','Pyrochroite[s].logact',
    'Rhodochrosite[s].logact','S.logact','S-2.logact','SO4-2.logact','S[+6].logact',
    'Si.logact','Si2O2[OH]5-.logact','Si2O3[OH]4-2.logact','Si3O5[OH]5-3.logact',
    'Si3O6[OH]3-3.logact','Si4O6[OH]6-2.logact','Si4O7[OH]6-4.logact',
    'Si4O8[OH]4-4.logact','Si6O15-6.logact','Siderite[s].logact','Sylvite[s].logact',
    'Talc[s].logact','Vivianite[s].logact',
]


out_con = [
    'Alkalinity.con','Anhydrite[s].con','Ar.con','Ar[g].con',
    'Aragonite[s].con','C.con','CO2.con','CO2[g].con','CO2g[s].con',
    'CO3-2.con','C[+4].con','Ca.con','Ca+2.con','CaCO3.con',
    'CaHCO3+.con','CaOH+.con','CaSO4.con','Ca[HPO4].con','Ca[OH]+.con',
    'Ca[SO4].con','Calcite[s].con','Cl.con','Cl-.con','Dolomite[s].con',
    'Fe.con','Fe+2.con','FeCO3.con','FeCl+.con','FeCl2.con','FeCl3-.con',
    'FeS[ppt][s].con','Fe[+2].con','Fe[CO3]2-2.con','Fe[H2PO4]+.con',
    'Fe[HPO4].con','Fe[HS]+.con','Fe[HS]2.con','Fe[NH3]+2.con',
    'Fe[NH3]2+2.con','Fe[NH3]4+2.con','Fe[OH]+.con','Fe[OH]2.con',
    'Fe[OH]3-.con','Fe[OH]4-2.con','Fe[SO4].con','Gypsum[s].con',
    'H.con','H+.con','H2CO3.con','H2O.con','H2O[g].con','H2S.con',
    'H2[PO4]-.con','H2[SiO4]-2.con','H3[PO4].con','H3[SiO4]-.con',
    'H4[SiO4].con','HCO3-.con','HPO4-2.con','HS-.con','HSO4-.con',
    'H[+1].con','Halite[s].con','Hydroxyapatite[s].con','K.con',
    'K+.con','KPO4-2.con','KSO4-.con','K[HPO4]-.con',
    'Mackinawite[s].con','Melanterite[s].con','Mg.con','Mg+2.con',
    'MgCO3.con','MgHCO3+.con','MgOH+.con','MgSO4.con','Mg[H2PO4]+.con',
    'Mg[H3SiO4]+.con','Mg[HPO4].con','Mg[NH3]+2.con','Mg[NH3]2+2.con',
    'Mg[NH3]3+2.con','Mg[NH3]4+2.con','Mg[PO4]-.con','Mn.con',
    'Mn+2.con','Mn2[OH]+3.con','Mn2[OH]3+.con','MnCl+.con','MnCl2.con',
    'MnCl3-.con','Mn[CO3].con','Mn[HCO3]+.con','Mn[HPO4].con',
    'Mn[HPO4]2-2.con','Mn[NH3]+2.con','Mn[NH3]2+2.con','Mn[OH]+.con',
    'Mn[OH]2.con','Mn[OH]3-.con','Mn[OH]4-2.con','Mn[SO4].con','NH3.con',
    'NH4+.con','Na.con','Na+.con','NaCO3-.con','NaH2PO4.con',
    'NaHCO3.con','NaPO4-2.con','NaSO4-.con','Na[HPO4]-.con','O.con',
    'OH-.con','O[-2].con','PO4-3.con','Pyrochroite[s].con',
    'Rhodochrosite[s].con','S.con','S-2.con','SO4-2.con','S[+6].con',
    'Si.con','Si2O2[OH]5-.con','Si2O3[OH]4-2.con','Si3O5[OH]5-3.con',
    'Si3O6[OH]3-3.con','Si4O6[OH]6-2.con','Si4O7[OH]6-4.con',
    'Si4O8[OH]4-4.con','Si6O15-6.con','Siderite[s].con','Sylvite[s].con',
    'Talc[s].con','Vivianite[s].con',
]

out_si_minerals = [
    'Anhydrite[s].si','Aragonite[s].si','CO2g[s].si','Calcite[s].si',
    'Dolomite[s].si','FeS[ppt][s].si','Gypsum[s].si','Halite[s].si',
    'Hydroxyapatite[s].si','Mackinawite[s].si','Melanterite[s].si',
    'Pyrochroite[s].si','Rhodochrosite[s].si','Siderite[s].si',
    'Sylvite[s].si','Talc[s].si','Vivianite[s].si',
]

out_tot_minerals = [
    'Anhydrite[s].tot','Aragonite[s].tot','CO2g[s].tot','Calcite[s].tot',
    'Dolomite[s].tot','FeS[ppt][s].tot','Gypsum[s].tot','Halite[s].tot',
    'Hydroxyapatite[s].tot','Mackinawite[s].tot','Melanterite[s].tot',
    'Pyrochroite[s].tot','Rhodochrosite[s].tot','Siderite[s].tot',
    'Sylvite[s].tot','Talc[s].tot','Vivianite[s].tot',
]
out_tot_gases = ['Ar[g].tot', 'CO2[g].tot',]

out_extra = ['chargebalance', 'I', 'totcharge']


# For the simulations with precipitation we need to add fixed_logact_CO2 to control the CO2[g].logactivity.
# Please note we need add sufficient CO2 to the system to allow the fixed_logact_CO2 to reach equilibrium,
# this is done by overuling the HCO3-.tot value. Please note Orchestra compensates the pH using the chargebalance.

InVars = [
    'fixed_logact_CO2','gas_type', 'gasvolume_fixed',
    'Ar[g].logact',
    'Ca+2.tot','Cl-.tot','Fe+2.tot','HCO3-.tot','K+.tot','Mg+2.tot',
    'Mn+2.tot','NH4+.tot','Na+.tot','PO4-3.tot','S-2.tot','SO4-2.tot',
    'Si.tot',
    'T','pH','watervolume'
]

InVars_diss = [
    'Ar.diss',
    'Ca+2.diss','Cl-.diss','Fe+2.diss','HCO3-.diss','K+.diss','Mg+2.diss',
    'Mn+2.diss','NH4+.diss','Na+.diss','PO4-3.diss','S-2.diss','SO4-2.diss',
    'Si.diss',
]

### Two types of scenarios 
In order to analyse the current state of the sample as determined by the chemical analysis, we require a simulation without any mineral precipitation, gas exchange and pH correction. In order to analyse how the sample will change if we move it to a different condition, we need to bea able to allow for mineral precipitation and gas-exchange. In this last case it is wise to allow the pH to compensate the charge balance.

This means we need to have two *chemistry.inp* files for our pyOrchestra calculations:
1. *chemistry_leachate_no_precip.inp* for the initial sample analysis;
2. *chemistry_leachate_precip.inp* for the two scenario calculations;

For both input files the set-up is identical, the only difference is that in the *no_precip* version, no minerals are allowed to precipitate and for the the *precip* version minerals have been selected that are allowed to precipitate. With the *no_precip* version, saturation indices for minerals can have values larger than 1, indicating supersaturation, for the *precip* version, all supersaturated minerals will be precipitated ensuring that the saturation indices are 0 or very close to 0.


### Question 1: Initial sample analysis
We start interpreting the chemical state of the initial sample. The setup of pyOrchestra is identical to the approach we used earlier in weeks 3.7 and 4.2. We only adapted some sections to match our current calculation.

In [15]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_leachate_no_precip.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    InVars1 = np.array(InVars)
    
    # We select the output from Orchestra we need to use
    # pleaste note that lists of strings can be concatenated using the + operator.
    out_list = (
        InVars + out_logact + out_si_minerals + 
        out_tot_minerals + out_tot_gases + out_con + 
        InVars_diss + out_extra
    )
        
    OutVars1 = np.array(out_list)
        

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)


Reading and expanding calculator new stylechemistry_leachate_no_precip.inp
Scanning file: chemistry_leachate_no_precip.inp
Scanning file: objects2025_THe.txt
Scanning file: chemistry_leachate_no_precip.inp
Scanning file: objects2025_THe.txt
Including file: chemistry_leachate_no_precip.inp
Scanning file: objects2025_THe.txt
0.106 sec.
	Reading variables .... 0.075 s
testing:
35:fixed_logact_CO2
36:gas_type
37:gasvolume_fixed
7:Ar[g].logact
9:Ca+2.tot
11:Cl-.tot
13:Fe+2.tot
16:HCO3-.tot
18:K+.tot
20:Mg+2.tot
22:Mn+2.tot
24:NH4+.tot
26:Na+.tot
28:PO4-3.tot
30:S-2.tot
32:SO4-2.tot
34:Si.tot
38:T
14:pH
39:watervolume
35:fixed_logact_CO2
36:gas_type
37:gasvolume_fixed
7:Ar[g].logact
9:Ca+2.tot
11:Cl-.tot
13:Fe+2.tot
16:HCO3-.tot
18:K+.tot
20:Mg+2.tot
22:Mn+2.tot
24:NH4+.tot
26:Na+.tot
28:PO4-3.tot
30:S-2.tot
32:SO4-2.tot
34:Si.tot
38:T
14:pH
39:watervolume
40:Alkalinity.logact
41:Anhydrite[s].logact
42:Ar.logact
43:Aragonite[s].logact
44:C.logact
45:CO2.logact
46:CO2[g].logact
47:CO2g[s].log

Please note that the output of this code is what ORCHESTRA echos back. ORCHESTRA uses a set of variables in order to store the input variables and the results of the calculations, in this case 33. The top part of the output shows the output requested by us through *OutVars* together with the values used during initialization.

### Run the Problem

In the above step we initialized our pyOrchestra class and made it available throug the *pO1* variable. In order to be able to run a calculation we need to update the primary states in the _InVars1_ variable with the molar concentrations from our sample and then run the calculations.

The data we will use is in the df_work dataframe. We can show this information in neicely formatted way with the following code.


In [16]:
# Let us print the initial primary state values from the database

table_md_data = df_work.sort_values(['orchestra_param'])[['orchestra_param','val_mol_l']].to_markdown()
display(Markdown(table_md_data))

# print(df_data)
#df_work.sort_values(['orchestra_param'])['orchestra_param'].to_list()

| cname              | orchestra_param   |     val_mol_l |
|:-------------------|:------------------|--------------:|
| Calcium [Ca]       | Ca+2.tot          |   0.0162176   |
| Chloride           | Cl-.tot           |   0.00705219  |
| IJzer [Fe]         | Fe+2.tot          |   0.000145031 |
| Bicarbonaat        | HCO3-.tot         |   0.0262209   |
| Kalium [K]         | K+.tot            |   0.00332481  |
| Magnesium [Mg]     | Mg+2.tot          |   0.00699301  |
| Mangaan [Mn]       | Mn+2.tot          |   2.1842e-05  |
| Ammonium (als NH4) | NH4+.tot          |   0.00336398  |
| Natrium [Na]       | Na+.tot           |   0.0126142   |
| Fosfaat (als PO4)  | PO4-3.tot         |   7.08928e-05 |
| Sulfide            | S-2.tot           |   1.74618e-06 |
| Sulfaat (als SO4)  | SO4-2.tot         |   0.0147333   |
| Silicium [Si]      | Si.tot            |   0.000509078 |
| Temperatuur        | T                 | 286.45        |
| pH                 | pH                |   6.81        |

In [17]:
# %%
# Run the initialize pyOrchestra class for the data in df_work

# Initialise the IN1 array and the output matrix (all_Res)
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
all_Res = np.zeros([1,len(OutVars1)])

# Initialize IN1 with the values in df_work. To do this we loop through the orchestra parameters
# present in df_work and add them to the correct position in IN1 (which is a NumPy array). The
# correct position is found using the InVars1 array and the param variable found in df_work.
# The position is found with the np.where method. The data is selected using the datafame.loc method.
for param in df_work['orchestra_param']:
    # print(param)
    sel_param = df_work['orchestra_param'] == param
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[sel_param, 'val_mol_l'].values[0]

# Not all values are available from df_work
# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = 0 # fixed CO2 logact
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = -20 #   # no gasvolume no background gas 
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1e-20 # no gas volume present

# run ORCHESTRA
OUT = pO1.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation = pd.DataFrame([all_Res],columns=OutVars1, index=['initial calculation'])


# Display some of the output in a nicely formatted way
# A first analysis will be on the chargebalance, to do this we require the charge balance
# and the total charge.
table_mdini = Res_Simulation[[
    'pH','chargebalance','totcharge','Ar[g].logact', 
    'CO2[g].logact','HCO3-.tot', 'HCO3-.logact', 
    'Ca+2.logact', 'Calcite[s].si', 'Gypsum[s].si',
    'HCO3-.con', 'CO3-2.con', 'Na+.diss']].to_markdown()
display(Markdown(table_mdini))


|                     |   pH |   chargebalance |   totcharge |   Ar[g].logact |   CO2[g].logact |   HCO3-.tot |   HCO3-.logact |   Ca+2.logact |   Calcite[s].si |   Gypsum[s].si |   HCO3-.con |   CO3-2.con |   Na+.diss |
|:--------------------|-----:|----------------:|------------:|---------------:|----------------:|------------:|---------------:|--------------:|----------------:|---------------:|------------:|------------:|-----------:|
| initial calculation | 6.81 |       0.0130261 |   0.0723384 |            -20 |       -0.980473 |   0.0262209 |       -1.91304 |      -2.37625 |        0.509987 |      -0.242832 |   0.0150015 |  6.8518e-06 |  0.0126142 |

### Interpretation of the initial analysis
For this interpretation we want to answer a number of questions:
1. Can we trust the analysis? To do this we can look a the charge balance and the electrical balance (which is 100*(charge balance/ total charge));
2. We can look at the saturation indices and the partial pressures of the gaseous species.

We first look at the charge balance:


In [18]:
# Calculate the Electrical Balance in % from the results
EB = Res_Simulation['chargebalance']/Res_Simulation['totcharge'] * 100

print(f"The chargebalance is {Res_Simulation['chargebalance'].values[0]} and the total charge in the system is {Res_Simulation['totcharge'].values[0]}. ")
print(f"The electrical balance is therefore {EB.values[0]} %. ")

The chargebalance is 0.013026062399148941 and the total charge in the system is 0.07233839482069016. 
The electrical balance is therefore 18.00712013244629 %. 


The electrical balance is about 0.67 % which is excellent which is an indication. When a sample is taken, it is split in to to fractions:
- for the cations, to which a strong acid is added to prevent precipitation of solids;
- for the anions. Here we cannot add acid, as this will lead to degassing of CO2.

An explanation for the slight positive charge balance can be the relatively high CO2[g] pressure in the sample compared to that in the atmosphere. A CO2[g].logact value of -2.27 corresponds with a partial pressure of CO2 of 10^(-2.27) = 5.37e-3 atm which is higher than the 0.420e-3 found in the atmosphere. The sample was therefore pressurized and as a consequence degassing may have occured during the analysis in the laboratory leading to a lower analysed concentration of HCO3-. As the degassing would have had less impact on the cation sample because it was acidified, you would expect a postive charge balance.

From the output above we see that the SI-value for Calcite is positive which is a strong indication that the sample is super saturated and not in an equilibrium state during sampling. 

Given the good charge balance of the analysis we move forward with the analysis where we need to be careful in trusting the carbonate measurements.

The next step in our interrpetation is list the minerals with SI-values that are larger than -0.5. These minerals may have been present in the wastebody where the sample originated and may precipitate if the sample is brought in to equilibrium with the atmosphere, for example in the water treatment plant.

In [19]:
# List all minerals with SI-values > 0.5

sel_large_SI = Res_Simulation.loc['initial calculation', out_si_minerals] > -0.5

# Display the results in a nice output format
selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))


|                     |   Aragonite[s].si |   Calcite[s].si |   Dolomite[s].si |   FeS[ppt][s].si |   Gypsum[s].si |   Hydroxyapatite[s].si |   Rhodochrosite[s].si |   Siderite[s].si |   Vivianite[s].si |
|:--------------------|------------------:|----------------:|-----------------:|-----------------:|---------------:|-----------------------:|----------------------:|-----------------:|------------------:|
| initial calculation |           0.30135 |        0.509987 |         0.718061 |        -0.102524 |      -0.242832 |                2.31875 |             -0.200098 |         0.630111 |           1.45632 |

Clearly quite a number of other minerals besides Calcite[s] are superstaturated. Understanding the composition of these minerals makes interpretation easier:
The compositions of the following minerals can be found using an internet search, or derived from the Orchestra input file.
- Calcite[s]: CaCO3
- Dolomite[s]: CaMg(CO3)2
- FeS[ppt][s]: FeS
- Hydroxyapatite[s]: Ca5(PO4)3(OH)
- Siderite[s]: FeCO3
- Talc[s]: Mg3Si4O10(OH)2
- Vivianite[s]: Fe3(PO4)2.8H2O

But as many of these minerals share master-species, it is not sure which of these minerals will truely precipitate or which have truely been present in the waste body. 

With this interpretation we have finalized our answer to the first question on the understanding the initial situation of the sample during sampling. 

To further understand this sample we can use pyOrchestra to assess the next 2 questions which give us information on how the sample changes when:
1. moving it to conditions occuring in the water treatment plant (in equilibrium with the atmosphere);
2. what the sample composition would have been if it were in the waste body (higher CO2[g] pressure and ubiquitous amounts of minerals present);



### Question 1: What happens when the sample is brought in equilibrium with the atmosphere

In order to answer this question we need to initialize a pyOrchestra class with the *chemistry_leachate_precip.inp* file. In this file minerals are allowed to precipitate and we can use the *fixed_logact_CO2* variable to fixe the CO2[g] pressure.

Initialization is similar as above, however the variable name is pO2.

In [20]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_leachate_precip.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars1 = np.array(InVars)
    
    # We select the output from Orchestra we need to use
    # We use the Output selector tab in the GUI to select the output variables.
    
    out_list = (
        InVars + out_logact + out_si_minerals + 
        out_tot_minerals + out_tot_gases + out_con + 
        InVars_diss + out_extra
    )
    
    OutVars1 = np.array(out_list)
        

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO2 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO2.initialise(InputFile, NoCells, InVars1, OutVars1)

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO2 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO2.initialise(InputFile, NoCells, InVars1, OutVars1)


Try a first calculation with iia switched off....
Parsing expressions of chemistry_leachate_no_precip.inp..... 
Optimizing expressions of chemistry_leachate_no_precip.inp..... 0.457 sec.
6837 variables, 20524 expressions, 14 equations.
First calculation was successful!
Repeat calculation with iia switched on..
Switching on: logI: -2
This was successful!!
Reading and expanding calculator new stylechemistry_leachate_precip.inp
Scanning file: chemistry_leachate_precip.inp
Scanning file: objects2025_THe.txt
Scanning file: chemistry_leachate_precip.inp
Scanning file: objects2025_THe.txt
Including file: chemistry_leachate_precip.inp
Scanning file: objects2025_THe.txt
0.106 sec.
	Reading variables .... 0.075 s
testing:
45:fixed_logact_CO2
46:gas_type
47:gasvolume_fixed
8:Ar[g].logact
10:Ca+2.tot
12:Cl-.tot
14:Fe+2.tot
17:HCO3-.tot
19:K+.tot
21:Mg+2.tot
23:Mn+2.tot
25:NH4+.tot
27:Na+.tot
29:PO4-3.tot
31:S-2.tot
33:SO4-2.tot
35:Si.tot
48:T
15:pH
49:watervolume
45:fixed_logact_CO2
46:gas_type
47

### Setting up the scenario
For this scenario we equilibrate with the atmosphere. We assume the partial pressure of CO2[g] to be 420 ppm, or CO2[g].logact = -3.38. The *chemistry_leachate_precip.inp* file already is prepared to allow all minerals to precipitate if supersaturated. The pH is set to balance the charge.

We follow the same approach as above, i.e. the molar concentrations for the master species in the InVars1 array come from df_work. However, as the atmosphere contains much more CO2[g] than present in our analysed sample, we need to add an exess amount of CO2 to our system so that the sample will be able to reach equilibrium with the set *fixed_logact_CO2* variable. We do this by setting the *HCO3-.tot* value to 10 mol/l. 

You need to increase the *HCO3-.tot* value if the CO2[g].logact remains smaller than the one defined by *fixed_logact_CO2*. In other words, the sample needs to be supersaturated with respect to the CO2[g] partial gas pressure to allow the model to achieve the targeted CO2[g] pressure.

```{note}
The Orchestra model system is organized in such a way that the excess amount of CO2[g] not required for reaching the equilibrium will be stored as a "virtual mineral" in the mineral phase. This mineral is labelled as *CO2g[s]* and we can evaluate its _*.tot_ value to see how much of the CO2[g] is not used in the reaction(s).
```
With this interpretation we are able to answer question 2.

In [21]:
# %%
# Run the model using chemistry_leachate_precip.inp in order 
# to move the selected sample to equilibrium with the atmosphere

#Initialise the input array
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars1)])

for param in df_work['orchestra_param']:
    # print(param)
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[df_work['orchestra_param'] == param, 'val_mol_l'].values[0]

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = -3.38 # fixed CO2 logact atmospheric CO2 pressure
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = -20 # background pressure = 0 atm due to gasvolume = 0
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1000 # gasvolume = 0

# We need to make sure that sufficient CO2 is present in the system to allow the system to reach the specified CO2[g].logact
# We add CO2 with CO3-2.tot
IN1[0][np.where(InVars1 == 'HCO3-.tot')] = 10 # Excess of CO2, will end up as precipitates 
# and in the virtual CO2g[s] mineral which serves as a store for CO2[g]

# run ORCHESTRA
OUT = pO2.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res and store this with an descriptive index.
Res_Simulation.loc['atmospheric equilibrium'] = all_Res


t : 0
206 : CO2[g].tot : 0
207 : Alkalinity.con : 0
208 : Anhydrite[s].con : 0
209 : Ar.con : 0
210 : Ar[g].con : 0
211 : Aragonite[s].con : 0
212 : C.con : 0
213 : CO2.con : 0
214 : CO2[g].con : 0
215 : CO2g[s].con : 0
216 : CO3-2.con : 0
217 : C[+4].con : 0
218 : Ca.con : 0
219 : Ca+2.con : 0
220 : CaCO3.con : 0
221 : CaHCO3+.con : 0
222 : CaOH+.con : 0
223 : CaSO4.con : 0
224 : Ca[HPO4].con : 0
225 : Ca[OH]+.con : 0
226 : Ca[SO4].con : 0
227 : Calcite[s].con : 0
228 : Cl.con : 0
229 : Cl-.con : 0
230 : Dolomite[s].con : 0
231 : Fe.con : 0
232 : Fe+2.con : 0
233 : FeCO3.con : 0
234 : FeCl+.con : 0
235 : FeCl2.con : 0
236 : FeCl3-.con : 0
237 : FeS[ppt][s].con : 0
238 : Fe[+2].con : 0
239 : Fe[CO3]2-2.con : 0
240 : Fe[H2PO4]+.con : 0
241 : Fe[HPO4].con : 0
242 : Fe[HS]+.con : 0
243 : Fe[HS]2.con : 0
244 : Fe[NH3]+2.con : 0
245 : Fe[NH3]2+2.con : 0
246 : Fe[NH3]4+2.con : 0
247 : Fe[OH]+.con : 0
248 : Fe[OH]2.con : 0
249 : Fe[OH]3-.con : 0
250 : Fe[OH]4-2.con : 0
251 : Fe[SO4].con : 0
2

We follow the same approach as above to interpret the results of this calculation. Because we allow the pH to be changed in order to ensure the charge balance, this will always be perfect (i.e. charge balance = 0).

The summary output for the same species above and be printed for both calculations.


In [22]:
table_mdini = Res_Simulation[[
    'T','pH', 'chargebalance','totcharge','Ar[g].logact', 'CO2[g].logact','HCO3-.tot', 'HCO3-.logact', 
    'Ca+2.logact', 'Calcite[s].si', 'Gypsum[s].si',
    'HCO3-.con', 'CO3-2.con']].to_markdown()
display(Markdown(table_mdini))

|                         |      T |      pH |   chargebalance |   totcharge |   Ar[g].logact |   CO2[g].logact |   HCO3-.tot |   HCO3-.logact |   Ca+2.logact |   Calcite[s].si |   Gypsum[s].si |   HCO3-.con |   CO3-2.con |
|:------------------------|-------:|--------:|----------------:|------------:|---------------:|----------------:|------------:|---------------:|--------------:|----------------:|---------------:|------------:|------------:|
| initial calculation     | 286.45 | 6.81    |       0.0130261 |   0.0723384 |            -20 |       -0.980473 |   0.0262209 |       -1.91304 |      -2.37625 |        0.509987 |      -0.242832 | 0.0150015   | 6.8518e-06  |
| atmospheric equilibrium | 286.45 | 8.02017 |       0         |   0.0582799 |            -20 |       -3.38     |  10         |       -3.1024  |      -2.90705 |        0        |      -0.586647 | 0.000945139 | 6.46029e-06 |

Clearly, allowing precipitation and bringing the sample to equilibrium with the CO2[g] in the atmosphere leads to significant changes especially  for the Ca+2 and HCO3- log activities. This is caused by degassing and precipitation of minerals. The targetted value for CO2[g].logact is indeed forced to -3.38. Apparently we have allowed for a sufficiently large surplus of carbonate in the system.

The pH of the leachate increased to 8.8, which indicates that the sample is also in equilibrium with solid carbonate minerals which buffer the pH to 8.8.

This output is only a small amount of the output available. We will now check which minerals have an SI value larger than -0.5. And we will display the amounts of precipitated minerals.

In [23]:
# List all minerals with SI-values > -0.5

sel_large_SI = Res_Simulation.loc['atmospheric equilibrium', out_si_minerals] >= -0.5

selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

# print(sel_large_SI)

sel_prec_min = Res_Simulation.loc['atmospheric equilibrium', out_tot_minerals] > 0

selected_minerals = sel_prec_min[sel_prec_min].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))


|                         |   Aragonite[s].si |   CO2g[s].si |   Calcite[s].si |   Dolomite[s].si |   FeS[ppt][s].si |   Hydroxyapatite[s].si |   Rhodochrosite[s].si |   Siderite[s].si |   Talc[s].si |
|:------------------------|------------------:|-------------:|----------------:|-----------------:|-----------------:|-----------------------:|----------------------:|-----------------:|-------------:|
| initial calculation     |          0.30135  |    -0.980473 |        0.509987 |         0.718061 |     -0.102524    |                2.31875 |             -0.200098 |         0.630111 |    -6.65981  |
| atmospheric equilibrium |         -0.208637 |     0        |        0        |         0        |     -1.77636e-15 |                0       |             -0.185721 |         0        |    -0.104756 |

|                         |   CO2g[s].tot |   Calcite[s].tot |   Dolomite[s].tot |   FeS[ppt][s].tot |   Hydroxyapatite[s].tot |   Siderite[s].tot |
|:------------------------|--------------:|-----------------:|------------------:|------------------:|------------------------:|------------------:|
| initial calculation     |       0       |        0         |        0          |       0           |              0          |       0           |
| atmospheric equilibrium |       9.56772 |        0.0080083 |        0.00298386 |       1.53181e-06 |              2.3326e-05 |       0.000109406 |

Clearly all minerals that were super-saturated during sampling, will precipitate when the sample is brought to equilibrium with the atmosphere. Interestingly we now also see Talc[s] precipitating, while this was not flagged as a super-saturated mineral in the initial sample.

The virtual mineral CO2g[s].tot indicates that 95.5 % of the carbonate added to the calculation as _HCO3-.tot_ has precipitated as CO2g[s]. This is a confirmation that we have included a sufficient amount of additional carbonate for the simulation to achieve the target log activity for CO2[g] of -3.38.


```{note}
The totals precipitated will end up as solids (sludge) in the water treatment system. Using the molar masses of the minerals you should be able to calculate how much solids will precipitate per liter sludge. This is an important result from this analysis as this provides the landfill operator essential information required to manage the water treatment plant. The sludge needs to be removed and treated.
```

### Question 2: What was the composition of the leachate within the waste body?
Using the above simulation results we can now use the model to make an educated guess of the leachate composition within the wastebody.

We do not have to initialize an extra pyOrchestra class because the *chemistry_leachate_precip.inp* file can handle this scenario. The simulation requires to dissolve a range of solid minerals present in the wastebody and equilibrate the CO2[g] pressure to about 0.5 atm. We can do this with the *fixed_logact_CO2* variable which is fixed to *log10(0.5)*.

We make two major assumptions for this calculation:
1. The CO2[g] pressure within the waste body is much higher than in the atmosphere. If we assume anaerobic conditions where methanogenic conditions occur we may estimate the CO[g] pressure to be 0.5 atm with a total pressure of about 1 atm.
2. The minerals which have relatively large SI-values will defnitely be present in the waste body.

We now need to adjust the master variables in such a way that we can have the model simulate the conditions in the waste body.

The compositions of the following minerals can be found from the input file of Orchestra and are:
- Calcite[s]: CaCO3
- Dolomite[s]: CaMg(CO3)2
- FeS[ppt][s]: FeS
- Hydroxyapatite[s]: Ca5(PO4)3(OH)
- Siderite[s]: FeCO3
- Talc[s]: Mg3Si4O10(OH)2
- Vivianite[s]: Fe3(PO4)2.8H2O

We assume that above minerals are present in the waste body in excess amounts (i.e. in concentrations larger than 10 moles per liter).
So we change the input values of the master-species as follows:

- Ca+2.tot == 10 + 10 + 50 = 70
- Fe+2.tot == 50 
- S-2.tot == 10
- Si.tot == 40
- PO4-3.tot == 50
- HCO3-.tot == 50 (also for atmosphere)!
- Mg+2.tot == 40

The CO2 pressure is equal to 0.5 atm, so fixed_log_act == -0.301.

Again the way we define the initial masses of the master-species is similar to the previous simulations. We start from the measured concentrations in df_work. Then we overule the above listed master species with the values given above. Because the minerals are allowed to precipitate, pyOrchestra will calculate the solution composition in equilibrium with the solid minerals. The excess amounts of master species will be found as mineral phases in the output.


In [24]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars1)])

for param in df_work['orchestra_param']:
    # print(param)
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[df_work['orchestra_param'] == param, 'val_mol_l'].values[0]

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = -0.301 # fixed CO2 logact atmospheric CO2 pressure
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = -20 # background pressure = 1 atm 
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1e-20 # 1 atm pressure

# We need to make sure that sufficient CO2 is present in the system to allow the system to reach the specified CO2[g].logact
# We add CO2 with CO3-2.tot

IN1[0][np.where(InVars1 == 'Ca+2.tot')] = 70
IN1[0][np.where(InVars1 == 'Fe+2.tot')] = 50 
IN1[0][np.where(InVars1 == 'Si.tot')] = 40
IN1[0][np.where(InVars1 == 'Mg+2.tot')] = 40

IN1[0][np.where(InVars1 == 'S-2.tot')] = 10
IN1[0][np.where(InVars1 == 'PO4-3.tot')] = 50
IN1[0][np.where(InVars1 == 'HCO3-.tot')] = 40 # Excess of CO2, will end up as precipitates 
# and in the virtual CO2g[s] mineral which serves as a store for CO2[g]


# run ORCHESTRA
OUT = pO2.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation.loc['wastebody equilibrium'] = all_Res


In [25]:
table_mdini = Res_Simulation[[
    'T','pH', 'chargebalance','totcharge','Ar[g].logact', 'CO2[g].logact','HCO3-.tot', 'HCO3-.logact', 
    'Ca+2.logact', 'Calcite[s].si', 'Gypsum[s].si',
    'HCO3-.con', 'CO3-2.con']].to_markdown()
display(Markdown(table_mdini))

|                         |      T |      pH |   chargebalance |   totcharge |   Ar[g].logact |   CO2[g].logact |   HCO3-.tot |   HCO3-.logact |   Ca+2.logact |   Calcite[s].si |   Gypsum[s].si |   HCO3-.con |   CO3-2.con |
|:------------------------|-------:|--------:|----------------:|------------:|---------------:|----------------:|------------:|---------------:|--------------:|----------------:|---------------:|------------:|------------:|
| initial calculation     | 286.45 | 6.81    |       0.0130261 |   0.0723384 |            -20 |       -0.980473 |   0.0262209 |       -1.91304 |      -2.37625 |        0.509987 |      -0.242832 | 0.0150015   | 6.8518e-06  |
| atmospheric equilibrium | 286.45 | 8.02017 |       0         |   0.0582799 |            -20 |       -3.38     |  10         |       -3.1024  |      -2.90705 |        0        |      -0.586647 | 0.000945139 | 6.46029e-06 |
| wastebody equilibrium   | 286.45 | 6.30855 |       0         |   0.129416  |            -20 |       -0.301    |  40         |       -1.73502 |      -2.57883 |       -0.016019 |      -0.398215 | 0.0226655   | 3.29036e-06 |

Clearly the hypothetical state of the leachate within the waste body is again different from the state of the leachate at sampling. The pH is lower, which is realate to the high partial pressure of CO2[g]. Calcite[s] seems to be subsaturated, albeit only slightly. For a beter insight we better check all SI-values close to zero and the amounts of solids that have precipitated (i.e. which are most likely to be present in the waste body). 

In [26]:
sel_prec_min[sel_prec_min]

CO2g[s].tot              True
Calcite[s].tot           True
Dolomite[s].tot          True
FeS[ppt][s].tot          True
Hydroxyapatite[s].tot    True
Siderite[s].tot          True
Name: atmospheric equilibrium, dtype: bool

In [27]:
# print the values where SI is > -0.5
sel_large_SI = Res_Simulation.loc['wastebody equilibrium', out_si_minerals] >=-0.5

selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

# print the values of the total mass of minerals in the system
sel_prec_min = Res_Simulation.loc['wastebody equilibrium', out_tot_minerals] > 0

selected_minerals = sel_prec_min[sel_prec_min].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

|                         |   Aragonite[s].si |   CO2g[s].si |   Calcite[s].si |   Dolomite[s].si |   FeS[ppt][s].si |   Gypsum[s].si |   Hydroxyapatite[s].si |   Rhodochrosite[s].si |   Siderite[s].si |   Talc[s].si |   Vivianite[s].si |
|:------------------------|------------------:|-------------:|----------------:|-----------------:|-----------------:|---------------:|-----------------------:|----------------------:|-----------------:|-------------:|------------------:|
| initial calculation     |          0.30135  | -0.980473    |        0.509987 |         0.718061 |     -0.102524    |      -0.242832 |            2.31875     |             -0.200098 |         0.630111 | -6.65981     |           1.45632 |
| atmospheric equilibrium |         -0.208637 |  0           |        0        |         0        |     -1.77636e-15 |      -0.586647 |            0           |             -0.185721 |         0        | -0.104756    |          -1.07973 |
| wastebody equilibrium   |         -0.224656 |  1.77636e-15 |       -0.016019 |         0        |     -8.88178e-16 |      -0.398215 |           -3.55271e-15 |             -0.321584 |         0        |  1.77636e-15 |           0       |

|                         |   CO2g[s].tot |   Dolomite[s].tot |   FeS[ppt][s].tot |   Hydroxyapatite[s].tot |   Siderite[s].tot |   Talc[s].tot |   Vivianite[s].tot |
|:------------------------|--------------:|------------------:|------------------:|------------------------:|------------------:|--------------:|-------------------:|
| initial calculation     |      0        |        0          |       0           |              0          |       0           |       0       |            0       |
| atmospheric equilibrium |      9.56772  |        0.00298386 |       1.53181e-06 |              2.3326e-05 |       0.000109406 |       0       |            0       |
| wastebody equilibrium   |      0.861342 |       10.0696     |       9.99997     |             11.9839     |      18.9278      |       9.97346 |            7.02405 |

These results are interesting to see. The pH is so low, that Calcite is completely dissolved, and the Ca+2 we associated with Calcite is now present in Hydroxyappatite. Remember Hydroxyapathite has 5 moles of Ca+2 per mole. The PO4-3 for the additional Hydroxyapathite comes from the amount we allocated to Vivianite. Roughly 0.85 moles of carbonate is buffered in the CO2g[s] used to ensure gas equilibrium. Apparently we did not require an excess of HCO3- to force the equilibrium with the *fixed_logact_CO2*. You can check this by setting the *HCO3-.tot* value in the simulation above to 40 mol/l instead of 50 mol/l. You then will see that the CO2g[s].tot value drops to the same value of 0.88. In these anaerobic, methanogenic systems the partial pressure is quite high, roughly 50% of the total pressure, so we may assume that there is sufficient excess CO2[g] present in the system.

Our aim was to understand how the composition of the leachate changes as it moves from the waste body to the drainage system and then to the water treatment plant. The composition of the leachate in the waste body can be found if we print the _*.diss_ values of the master species which we already create a list for above: InVars_diss.

In [28]:
table_mdini = Res_Simulation[InVars_diss].to_markdown()
display(Markdown(table_mdini))


|                         |     Ar.diss |   Ca+2.diss |   Cl-.diss |   Fe+2.diss |   HCO3-.diss |    K+.diss |   Mg+2.diss |   Mn+2.diss |   NH4+.diss |   Na+.diss |   PO4-3.diss |    S-2.diss |   SO4-2.diss |     Si.diss |
|:------------------------|------------:|------------:|-----------:|------------:|-------------:|-----------:|------------:|------------:|------------:|-----------:|-------------:|------------:|-------------:|------------:|
| initial calculation     | 9.99995e-41 |  0.0162176  | 0.00705219 | 0.000145031 |   0.0262209  | 0.00332481 |  0.00699301 |  2.1842e-05 |  0.00336398 |  0.0126142 |  7.08928e-05 | 1.74618e-06 |    0.0147333 | 0.000509078 |
| atmospheric equilibrium | 9.99995e-41 |  0.00510878 | 0.00705219 | 3.40933e-05 |   0.00105505 | 0.00332481 |  0.00400915 |  2.1842e-05 |  0.00336398 |  0.0126142 |  9.14724e-07 | 2.14371e-07 |    0.0147333 | 0.000509078 |
| wastebody equilibrium   | 9.99995e-41 |  0.0109036  | 0.00705219 | 7.09857e-05 |   0.0716021  | 0.00332481 |  0.00998377 |  2.1842e-05 |  0.00336398 |  0.0126142 |  0.000217137 | 2.77286e-05 |    0.0147333 | 0.106144    |

### Question 3 How much precipitation of solids will occur from the leachate as it moves from the wastebody to the water treatment system?
In order to get a better understanding the differences between the three types of leachate and facilitate a proces based interpretation we need to carry out one more pyOrchestra calculation. With the "wastebody equilibrium" result we have a quantification of a likely composition of the leachate. It is this leachate that moves to the drainage system and then from there to the waste water treatment plant. What we now can do is to see how this wastebody leachate changes if we move it to the water treatment plant. Will the composition of the sample resemble the one we calculated using the composition of the sampled leachate?

To do this equation we use the _*.diss_ values which are the total molar concentrations of the master species in the dissolved phase. This is logical because we are only moving the leachate, while we are leaving the solids and gas behind. 


In [29]:
Res_Simulation.loc['wastebody equilibrium',InVars_diss+['pH']]

Ar.diss       9.999946e-41
Ca+2.diss     1.090364e-02
Cl-.diss      7.052186e-03
Fe+2.diss     7.098574e-05
HCO3-.diss    7.160213e-02
K+.diss       3.324808e-03
Mg+2.diss     9.983770e-03
Mn+2.diss     2.184201e-05
NH4+.diss     3.363978e-03
Na+.diss      1.261418e-02
PO4-3.diss    2.171368e-04
S-2.diss      2.772862e-05
SO4-2.diss    1.473331e-02
Si.diss       1.061435e-01
pH            6.308549e+00
Name: wastebody equilibrium, dtype: float32

In [30]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples
# IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars1)])


# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = -3.38 # fixed CO2 logact atmospheric CO2 pressure
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = -20 # background pressure = 1 atm 
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1e-20 # 1 atm pressure

# We need to make sure that sufficient CO2 is present in the system to allow the system to reach the specified CO2[g].logact
# We add CO2 with CO3-2.tot
IN1[0][np.where(InVars1 == 'pH')] = 7
IN1[0][np.where(InVars1 == 'Ca+2.tot')] = Res_Simulation.loc['wastebody equilibrium', 'Ca+2.diss']
IN1[0][np.where(InVars1 == 'Cl-.tot')] = Res_Simulation.loc['wastebody equilibrium', 'Cl-.diss']
IN1[0][np.where(InVars1 == 'Fe+2.tot')] = Res_Simulation.loc['wastebody equilibrium', 'Fe+2.diss']
IN1[0][np.where(InVars1 == 'K+.tot')] = Res_Simulation.loc['wastebody equilibrium', 'K+.diss']
IN1[0][np.where(InVars1 == 'Mg+2.tot')] = Res_Simulation.loc['wastebody equilibrium', 'Mg+2.diss']
IN1[0][np.where(InVars1 == 'Mn+2.tot')] = Res_Simulation.loc['wastebody equilibrium', 'Mn+2.diss']
IN1[0][np.where(InVars1 == 'Na+.tot')] = Res_Simulation.loc['wastebody equilibrium', 'Na+.diss']
IN1[0][np.where(InVars1 == 'NH4+.tot')] = Res_Simulation.loc['wastebody equilibrium', 'NH4+.diss']
IN1[0][np.where(InVars1 == 'PO4-3.tot')] = Res_Simulation.loc['wastebody equilibrium', 'PO4-3.diss']
IN1[0][np.where(InVars1 == 'S-2.tot')] = Res_Simulation.loc['wastebody equilibrium', 'S-2.diss']
IN1[0][np.where(InVars1 == 'Si.tot')] = Res_Simulation.loc['wastebody equilibrium', 'Si.diss']
IN1[0][np.where(InVars1 == 'SO4-2.tot')] = Res_Simulation.loc['wastebody equilibrium', 'SO4-2.diss']

IN1[0][np.where(InVars1 == 'HCO3-.tot')] = 5 # Excess carbonate to allow the fixed_logact_CO2 to be reached

# run ORCHESTRA
OUT = pO2.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation.loc['wastebody to atm'] = all_Res


In [31]:
# print the values where SI is > -0.5
sel_large_SI = Res_Simulation.loc['wastebody to atm', out_si_minerals] >=-0.5

selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

# print the values of the total mass of minerals in the system
sel_prec_min = Res_Simulation.loc['wastebody to atm', out_tot_minerals] > 0

selected_minerals = sel_prec_min[sel_prec_min].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

display(
    Markdown(
        Res_Simulation[InVars_diss+['pH']]
        .to_markdown()
    )
)


|                         |   Aragonite[s].si |   CO2g[s].si |   Calcite[s].si |   FeS[ppt][s].si |   Gypsum[s].si |   Hydroxyapatite[s].si |   Rhodochrosite[s].si |   Siderite[s].si |   Talc[s].si |
|:------------------------|------------------:|-------------:|----------------:|-----------------:|---------------:|-----------------------:|----------------------:|-----------------:|-------------:|
| initial calculation     |          0.30135  | -0.980473    |     0.509987    |     -0.102524    |      -0.242832 |            2.31875     |             -0.200098 |         0.630111 | -6.65981     |
| atmospheric equilibrium |         -0.208637 |  0           |     0           |     -1.77636e-15 |      -0.586647 |            0           |             -0.185721 |         0        | -0.104756    |
| wastebody equilibrium   |         -0.224656 |  1.77636e-15 |    -0.016019    |     -8.88178e-16 |      -0.398215 |           -3.55271e-15 |             -0.321584 |         0        |  1.77636e-15 |
| wastebody to atm        |         -0.208637 |  0           |    -4.44089e-16 |      8.88178e-16 |      -0.300061 |            0           |             -0.282804 |        -0.175133 |  0           |

|                         |   CO2g[s].tot |   Calcite[s].tot |   FeS[ppt][s].tot |   Hydroxyapatite[s].tot |   Talc[s].tot |
|:------------------------|--------------:|-----------------:|------------------:|------------------------:|--------------:|
| initial calculation     |      0        |      0           |       0           |             0           |    0          |
| atmospheric equilibrium |      9.56772  |      0.0080083   |       1.53181e-06 |             2.3326e-05  |    0          |
| wastebody equilibrium   |      0.861342 |      0           |       9.99997     |            11.9839      |    9.97346    |
| wastebody to atm        |      4.99917  |      9.16805e-05 |       2.74878e-05 |             7.22432e-05 |    0.00332479 |

|                         |     Ar.diss |   Ca+2.diss |   Cl-.diss |   Fe+2.diss |   HCO3-.diss |    K+.diss |   Mg+2.diss |   Mn+2.diss |   NH4+.diss |   Na+.diss |   PO4-3.diss |    S-2.diss |   SO4-2.diss |     Si.diss |      pH |
|:------------------------|------------:|------------:|-----------:|------------:|-------------:|-----------:|------------:|------------:|------------:|-----------:|-------------:|------------:|-------------:|------------:|--------:|
| initial calculation     | 9.99995e-41 |  0.0162176  | 0.00705219 | 0.000145031 |  0.0262209   | 0.00332481 | 0.00699301  |  2.1842e-05 |  0.00336398 |  0.0126142 |  7.08928e-05 | 1.74618e-06 |    0.0147333 | 0.000509078 | 6.81    |
| atmospheric equilibrium | 9.99995e-41 |  0.00510878 | 0.00705219 | 3.40933e-05 |  0.00105505  | 0.00332481 | 0.00400915  |  2.1842e-05 |  0.00336398 |  0.0126142 |  9.14724e-07 | 2.14371e-07 |    0.0147333 | 0.000509078 | 8.02017 |
| wastebody equilibrium   | 9.99995e-41 |  0.0109036  | 0.00705219 | 7.09857e-05 |  0.0716021   | 0.00332481 | 0.00998377  |  2.1842e-05 |  0.00336398 |  0.0126142 |  0.000217137 | 2.77286e-05 |    0.0147333 | 0.106144    | 6.30855 |
| wastebody to atm        | 9.99995e-41 |  0.0104507  | 0.00705219 | 4.3498e-05  |  0.000733937 | 0.00332481 | 9.40983e-06 |  2.1842e-05 |  0.00336398 |  0.0126142 |  4.07165e-07 | 2.40855e-07 |    0.0147333 | 0.0928444   | 7.85295 |

When we compare the data of the second simulation (atmospheric equilibrium) with the last (wastebody to atm) we see that the data are in the same order of magnitude but that the last concentrations are different. This is not surprising. Can you list a number of processes that are not taken in to account in this modelling approach but will have an impact on the estimated concentrations. 


## Day 3: Assessing the temporal variation in the data

### Question 3 How much precipitation of solids will occur from the leachate as it moves from the wastebody to the water treatment system?
On the first two days you looked at the data and performed a geochemical analysis on a selected sample. The task at hand for the third day is to check the temporal variation in the data set. As you well know, a year has four seasons, winter, spring, summer and autumn. As mentioned earlier, leachate production rates depend heavily on the season. In spring, after the winter with low evapo-transpiration, leachate production rates are highest, whereas in the autumn after summer with high evapo-transpiration leachate production rates tend to be lower. On this day you are provided with a data set with the measured cumulative leachate and leachate production rates.

The task at hand for you is to select some representative samples from the anlysed samples and repeat the analysis you did on the 2nd day. The samples should be selected in such a way that you are able to get an understanding of the seasonal variation in the data.

You should now have sufficient understanding how to code the analysis, that using copy-paste-edit you should be able to setup your analysis yourself. 

### Project actvities Day 3
The steps you need to do on this day are:
1. Import and plot the data from the leachate production data-set;
2. Use this information to choose time periods from which you want to analyse the geochemistry of the leachate;
3. Select specific samples, please ensure that the samples have sufficient parameters. Ensure use the time series of the chemical parameters to make an educated guess for the missing values;
4. Run three types of pyOrchetra calcautions to answer the three questions:
    - what is the initial state of the sample?, 
    - what happens if it is equilibrated with the atmosphere?, 
    - and what were the conditions where it originated? 
5. Consider putting the above steps in a single function which you can then call from a loop;
6. Interpret the results;
7. Include the outcomes of your work in your report;


## Day 4: Finalize your report

### Write a report using the report template provided to you

Every detailed analysis of a dataset needs to be reported in a consistent way. The aim of the report is to present your results, to motivate and document the strategy you applied in order to interpret the data and explain the assumputions you made while interpreting the data. In your report, you state the research questions and application challenges that need to solved. Finally you answer the questions using the results of your interpretation and your understanding of the underlying processes.

The abstract of your report is a concise summary of the above.

With this report you have concluded your group project.

